---
# PHẦN MỚI · Nâng Step 2 (đánh giá hiệu quả từng đoạn lập luận)

**Vì sao có notebook này.** Step 2 hiện yếu: macro-F1 0,43 (baseline), 0,477 theo đề sau khi chỉnh trọng số lớp; F1 lớp *Ineffective* chỉ 0,07 → 0,126. Mục tiêu của buổi họp là *trên 0,5*. Notebook này thử **3 nâng cấp** trên đúng cách chia cũ (leave-one-prompt-out, 8 đề independent):

| # | Nâng cấp | Block |
|---|---|---|
| 1 | Đo bằng **thước xếp hạng** (PR-AUC, precision/recall/lift trong top 5%) – đúng mục tiêu *ưu tiên cái giáo viên cần xem* | 4, 8 |
| 2 | **Embedding câu** (MiniLM) thay cho TF-IDF | 5, 7 |
| 3 | **Ngữ cảnh văn bản**: độ tương đồng đoạn với câu *Position*, với đoạn liền trước, với trung bình cả bài | 6, 7 |

**Cách chạy:** Run all, chạy từ thư mục `AFP_Project`. Đặt `QUICK = True` ở Block 1 để chạy thử nhanh; `False` = đầy đủ (khoảng 30–60 phút; embedding được lưu cache nên lần sau không tính lại).

**Quy tắc chống rò rỉ:** (i) mô hình bị fit **không có đề giữ lại**; (ii) trọng số lớp chỉ được chỉnh trên dự đoán out-of-fold của 7 đề huấn luyện; (iii) embedding dùng mô hình **đóng băng** (không học nhãn), PCA fit không nhãn.

**Lưu ý trung thực:** kết quả có thể *không* tăng nhiều – nhãn Ineffective chiếm ~1,9% và người chấm chỉ đồng thuận κ = 0,32. Báo cáo cả khi không tăng.

## Step 2 · §1. Cấu hình

In [ ]:
# ===== BLOCK S2-1 =====
import os, sys, time, warnings, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, average_precision_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50)

# >>> CHỈNH Ở ĐÂY <<<
QUICK = False        # True = chạy thử nhanh (120 bài/đề, 2 tiến trình); False = đầy đủ như báo cáo
N_JOBS = 4           # số đề chạy song song (mỗi tiến trình dùng 4 luồng)
TOP = 0.05           # tỉ lệ đoạn "đáng ngờ nhất" để tính precision/recall/lift
# ----------------------
os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists(): ROOT = ROOT.parent
PROC, RES, FIGS = ROOT / "data" / "processed", ROOT / "results", ROOT / "figs"
for d in (PROC, RES, FIGS): d.mkdir(parents=True, exist_ok=True)
LABELS = ["Adequate", "Effective", "Ineffective"]
try:
    import sentence_transformers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
print("ROOT:", ROOT, "| QUICK =", QUICK)

## Step 2 · §2. Dữ liệu: từ file gốc đến bảng đoạn dùng cho Step 2

Mỗi **dòng** = một đoạn lập luận (`discourse_text`) có nhãn vai trò (`discourse_type`) và nhãn hiệu quả (`discourse_effectiveness`: Ineffective / Adequate / Effective). Chỉ giữ **8 đề independent** và bỏ các đoạn *Unannotated* (không có nhãn hiệu quả) cùng 3 đoạn thiếu nhãn.
Từ đoạn và bài chứa nó, ta tính 3 nhóm đặc trưng dày (giải thích ở Block 3).

In [ ]:
# ===== BLOCK S2-2 =====
seg = pd.read_pickle(PROC / "segments.pkl")
ess = pd.read_pickle(PROC / "essays.pkl")
ind = ess[ess.task == "Independent"].reset_index(drop=True)          # 7.869 bài / 8 đề; đề source-based bị loại
if (PROC / "features_independent.pkl").exists():
    X1 = pd.read_pickle(PROC / "features_independent.pkl").fillna(0)   # 43 đặc trưng ngôn ngữ cấp bài (M1)
else:
    sys.path.insert(0, str(ROOT / "src")); from features import build_feature_frame
    X1 = build_feature_frame(ind.full_text).fillna(0); X1.to_pickle(PROC / "features_independent.pkl")
X1.insert(0, "essay_id_comp", ind.essay_id_comp.values)

seg = seg[seg.task == "Independent"].merge(ind[["essay_id_comp", "full_text"]], on="essay_id_comp")
seg = seg.sort_values(["essay_id_comp", "discourse_start"]).reset_index(drop=True)   # thứ tự xuất hiện trong bài
seg["role"] = seg.discourse_type
g = seg.groupby("essay_id_comp")
seg["rank"] = g.cumcount(); seg["n_all"] = g.discourse_start.transform("count")
seg["prev_role"] = g.role.shift(1).fillna("START"); seg["next_role"] = g.role.shift(-1).fillna("END")
seg["prev_role2"] = g.role.shift(2).fillna("START")
seg = seg[(seg.role != "Unannotated") & seg.discourse_effectiveness.notna()].copy()
g = seg.groupby("essay_id_comp")
seg["n_in_essay"] = g.discourse_start.transform("count")
seg["rank_rel"] = seg["rank"] / seg.n_all
seg["is_first"] = (seg["rank"] == 0) * 1.0; seg["is_last"] = (seg["rank"] == seg.n_all - 1) * 1.0
seg["rel_pos"] = seg.discourse_start / seg.full_text.str.len()
t = seg.discourse_text; tl = t.str.lower()
seg["log_len"] = np.log1p(seg.seg_len); seg["n_chars"] = t.str.len()
seg["essay_words"] = seg.full_text.str.split().str.len(); seg["len_share"] = seg.seg_len / seg.essay_words
seg["n_sent"] = t.str.count(r"[.!?]+") + 1; seg["avg_wlen"] = seg.n_chars / seg.seg_len.clip(lower=1)
seg["n_comma"] = t.str.count(","); seg["caps"] = t.str.count(r"[A-Z]") / seg.n_chars.clip(lower=1)
seg["has_digit"] = t.str.contains(r"\d") * 1.0
seg["i_cnt"] = tl.str.count(r"\bi\b") / seg.seg_len.clip(lower=1); seg["you_cnt"] = tl.str.count(r"\byou\b") / seg.seg_len.clip(lower=1)
seg["lower_start"] = t.str.strip().str[:1].str.islower() * 1.0
seg["no_end"] = (~t.str.strip().str[-1:].isin([".", "!", "?"])) * 1.0
seg = seg.merge(X1.add_prefix("e_").rename(columns={"e_essay_id_comp": "essay_id_comp"}), on="essay_id_comp")
if QUICK:                                                       # chỉ để thử code nhanh
    keep = seg.groupby("prompt_name").essay_id_comp.transform(lambda s: s.isin(s.unique()[:120]))
    seg = seg[keep]
seg = seg.reset_index(drop=True)
ESS_COLS = [c for c in seg.columns if c.startswith("e_") and c != "essay_words"]
y = seg.discourse_effectiveness.values; gp = seg.prompt_name.values; prompts = sorted(set(gp))
print("đoạn:", len(seg), "| bài:", seg.essay_id_comp.nunique(), "| đề:", len(prompts))
print(pd.Series(y).value_counts().to_frame("n").assign(share=lambda d: (d.n / d.n.sum()).round(3)))
print("\nTỉ lệ Ineffective theo đề:"); print(pd.Series(y == "Ineffective", index=gp).groupby(level=0).mean().round(3).to_string())

## Step 2 · §3. Ba nhóm đặc trưng dày (không dùng văn bản)

* **Vai trò** (one-hot): Lead, Position, Claim, Counterclaim, Rebuttal, Evidence, Concluding – *gold*, giống cách Nhóm 6 dùng `discourse_type`.
* **Độ dài / vị trí / bề mặt đoạn**: số từ, ký tự, câu, vị trí tương đối trong bài, là đoạn đầu hay cuối, dấu phẩy, chữ số, chữ "I"/"you", viết thường đầu đoạn, thiếu dấu kết…
* **Ngữ cảnh bài**: vai trò đoạn trước/sau, tỉ lệ độ dài đoạn so với bài, **43 đặc trưng ngôn ngữ cấp bài của M1** (để mô hình biết bài này nói chung viết tốt hay yếu).

Đây là khối **S3**, mức nền không có tín hiệu văn bản.

In [ ]:
# ===== BLOCK S2-3 =====
SEG_STATS = ["rank_rel","is_first","is_last","rel_pos","log_len","n_chars","seg_len","n_sent","avg_wlen","n_comma","caps",
             "has_digit","i_cnt","you_cnt","lower_start","no_end"]
CONTEXT = ["n_in_essay", "len_share", "essay_words"]
roles = pd.get_dummies(seg.role).astype(float)
nb = pd.concat([pd.get_dummies(seg.prev_role, prefix="prev"), pd.get_dummies(seg.next_role, prefix="next"),
                pd.get_dummies(seg.prev_role2, prefix="prev2")], axis=1).astype(float)
S3 = pd.concat([roles, seg[SEG_STATS], seg[CONTEXT], nb, seg[ESS_COLS]], axis=1)
print("S3:", S3.shape)
print("\nTrung bình vài đặc trưng theo nhãn hiệu quả (đoạn tốt có dài hơn không?):")
display(seg.groupby("discourse_effectiveness")[["seg_len","n_sent","rank_rel","len_share"]].mean().round(3))

## Step 2 · §4. Hàm đo: macro-F1, **thước xếp hạng**, chỉnh trọng số lớp

* **macro-F1**: trung bình F1 của 3 lớp.
* **PR-AUC (lớp Ineffective)**: xếp các đoạn theo xác suất Ineffective rồi đo chất lượng xếp hạng; *mức đoán bừa bằng tỉ lệ Ineffective (~1,9%)*.
* **precision/recall/lift @ top 5%**: nếu chỉ gắn cờ 5% đoạn đáng ngờ nhất của mỗi đề, bao nhiêu % thật sự là Ineffective, bắt được bao nhiêu % tổng số, và gấp mấy lần đoán bừa (`lift = precision / tỉ lệ nền`).
* **Trọng số lớp** `w = (1, a, b)` nhân vào xác suất trước `argmax`; `a, b` chọn trên **dự đoán out-of-fold của 7 đề huấn luyện**, không nhìn đề giữ lại.

In [ ]:
# ===== BLOCK S2-4 =====
def hgb(seed=0):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=6, l2_regularization=1.0, random_state=seed)

def tune_weights(P, yy):
    best, bw = -1, (1.0, 1.0, 1.0)
    for a in np.linspace(0.5, 5, 10):
        for b in np.linspace(0.5, 40, 40):
            s = f1_score(yy, np.array(LABELS)[(P * np.array([1, a, b])).argmax(1)], average="macro", labels=LABELS)
            if s > best: best, bw = s, (1.0, a, b)
    return np.array(bw)

def rank_metrics(p_ineff, yy):
    pos = (yy == "Ineffective"); k = max(int(round(TOP * len(yy))), 1); top = np.argsort(-p_ineff)[:k]
    prec = pos[top].mean()
    return {"PR_AUC": average_precision_score(pos, p_ineff), "prec_top5": prec,
            "recall_top5": pos[top].sum() / max(pos.sum(), 1), "lift_top5": prec / max(pos.mean(), 1e-9), "base_rate": pos.mean()}

def label_metrics(yy, pred):
    per = [f1_score(yy[gp == p], pred[gp == p], average="macro", labels=LABELS, zero_division=0) for p in prompts]
    pr, rc, f1, _ = precision_recall_fscore_support(yy, pred, labels=LABELS, zero_division=0)
    return {"macroF1_pooled": f1_score(yy, pred, average="macro", labels=LABELS, zero_division=0),
            "macroF1_mean_prompts": np.mean(per), "SD_prompts": np.std(per, ddof=1),
            "F1_Ineffective": f1[2], "P_Ineffective": pr[2], "R_Ineffective": rc[2]}
print("đã định nghĩa hàm đo")

## Step 2 · §5. Nâng cấp 2 – embedding câu (MiniLM, mô hình đóng băng)

`all-MiniLM-L6-v2` biến mỗi đoạn thành vector 384 chiều nắm *nghĩa* (TF-IDF chỉ đếm từ). Mô hình **không được huấn luyện lại bằng nhãn của ta**, nên không có rò rỉ nhãn. Lần đầu tính mất vài phút đến vài chục phút trên CPU; kết quả lưu vào `data/processed/seg_emb*.npz`.

In [ ]:
# ===== BLOCK S2-5 =====
cache = PROC / ("seg_emb_quick.npz" if QUICK else "seg_emb.npz")
E = None
if cache.exists():
    z = np.load(cache, allow_pickle=True)
    if len(z["ids"]) == len(seg) and (z["ids"] == seg.discourse_id.values).all(): E = z["E"]; print("nạp cache", cache.name)
if E is None:
    from sentence_transformers import SentenceTransformer
    t0 = time.time()
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    E = model.encode(seg.discourse_text.tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True).astype("float32")
    np.savez_compressed(cache, ids=seg.discourse_id.values, E=E); print(f"embedding {len(seg)} đoạn: {time.time()-t0:.0f}s")
print("E:", E.shape)

## Step 2 · §6. Nâng cấp 3 – ngữ cảnh văn bản + PCA

Ba độ tương đồng cosine (vector đã chuẩn hoá):

* `sim_position`: đoạn này liên quan thế nào tới **câu Position** (luận điểm chính) của bài.
* `sim_prev`: liên quan thế nào tới **đoạn liền trước** (có mạch lạc không).
* `sim_essay_mean`: nằm gần hay xa **trọng tâm** của cả bài (lạc đề?).

Embedding 384 chiều được nén còn **64 chiều bằng PCA** (không nhãn) cho gọn.

In [ ]:
# ===== BLOCK S2-6 =====
eid = seg.essay_id_comp.values; n = len(seg)
prev = np.zeros(n); same = np.r_[False, eid[1:] == eid[:-1]]
prev[same] = (E[1:] * E[:-1]).sum(1)[same[1:]]
cent = pd.DataFrame(E).groupby(eid).transform("mean").values
cent /= np.linalg.norm(cent, axis=1, keepdims=True) + 1e-9
is_pos = (seg.role == "Position").values
pos_idx = pd.Series(np.arange(n))[is_pos].groupby(eid[is_pos]).first()
pi = pd.Series(eid).map(pos_idx); has_pos = pi.notna().values
to_pos = np.zeros(n); to_pos[has_pos] = (E[has_pos] * E[pi[has_pos].astype(int).values]).sum(1)
sims = pd.DataFrame({"sim_prev": prev, "sim_essay_mean": (E * cent).sum(1), "sim_position": to_pos, "has_position": has_pos * 1.0})
Z = pd.DataFrame(PCA(64, random_state=0).fit_transform(E), columns=[f"emb{i}" for i in range(64)])
print("Độ tương đồng trung bình theo nhãn hiệu quả (đoạn Effective có bám luận điểm hơn không?):")
display(sims.assign(eff=y)[sims.has_position.values == 1].groupby("eff")[["sim_position","sim_prev","sim_essay_mean"]].mean().round(3))
BLOCKS = {"S3 no text signal": S3, "S5 +embedding": pd.concat([S3, Z], axis=1),
          "S6 +embedding +context sims": pd.concat([S3, Z, sims], axis=1)}
print({k: v.shape for k, v in BLOCKS.items()})

## Step 2 · §7. Chạy leave-one-prompt-out

Với mỗi đề giữ lại `p`: fit trên 7 đề còn lại để lấy xác suất cho `p`; đồng thời fit 7 lần *trong* 7 đề huấn luyện (bỏ lần lượt từng đề) để lấy dự đoán out-of-fold và chọn trọng số lớp. Có 3 khối đặc trưng × 8 đề × 8 lần fit.

In [ ]:
# ===== BLOCK S2-7 =====
def one_fold(p):
    t0 = time.time(); te = gp == p; tr = ~te; trp = sorted(set(gp[tr])); out = {}
    for name, X in BLOCKS.items():
        P_te = hgb().fit(X[tr], y[tr]).predict_proba(X[te])
        Xtr, ytr, gtr = X[tr], y[tr], gp[tr]; P_in = np.zeros((int(tr.sum()), 3))
        for q in trp:
            iq = gtr == q; P_in[iq] = hgb().fit(Xtr[~iq], ytr[~iq]).predict_proba(Xtr[iq])
        out[name] = (P_te, tune_weights(P_in, ytr))
    print(f"{p:40s} xong sau {time.time()-t0:.0f}s", flush=True)
    return p, out

t0 = time.time()
RESULT = dict(Parallel(n_jobs=2 if QUICK else N_JOBS)(delayed(one_fold)(p) for p in prompts))
print(f"tổng {(time.time()-t0)/60:.1f} phút")

## Step 2 · §8. Bảng kết quả

Đọc: (1) so `macroF1_*` và `F1_Ineffective` giữa các khối; (2) **so `PR_AUC` và `prec_top5` với `base_rate`** – nếu chỉ hơn mức nền chút ít thì cờ chưa hữu ích; (3) `lift_top5` = gấp mấy lần đoán bừa. Các cột xếp hạng là trung bình 8 đề.

In [ ]:
# ===== BLOCK S2-8 =====
rows = []
for name in BLOCKS:
    for tuned in (False, True):
        pred = np.empty(len(y), dtype=object); pi_ = np.zeros(len(y))
        for p in prompts:
            P, w = RESULT[p][name]; m = gp == p
            pred[m] = np.array(LABELS)[(P * (w if tuned else 1)).argmax(1)]; pi_[m] = P[:, 2]
        rk = [rank_metrics(pi_[gp == p], y[gp == p]) for p in prompts]
        rows.append({"model": name, "weights": "tuned" if tuned else "argmax", **label_metrics(y, pred),
                     **{k: np.mean([r[k] for r in rk]) for k in rk[0]}})
TAB = pd.DataFrame(rows).round(3)
TAB.to_csv(RES / ("step2_embed_summary_quick.csv" if QUICK else "step2_embed_summary.csv"), index=False)
display(TAB[["model","weights","macroF1_pooled","macroF1_mean_prompts","F1_Ineffective","PR_AUC","prec_top5","recall_top5","lift_top5","base_rate"]])
print("Mốc tham chiếu (báo cáo Tuần 9, baseline LR TF-IDF): macroF1 gộp 0,433 | F1 Ineffective 0,07")
print("Mốc 06_step2_improved (S4, chỉnh trọng số): macroF1 gộp 0,510 | theo đề 0,477 | F1 Ineffective 0,126")

### Step 2 · §8b. Theo từng đề (khối cuối, trọng số đã chỉnh)

In [ ]:
# ===== BLOCK S2-9 =====
last = list(BLOCKS)[-1]; rows = []
for p in prompts:
    P, w = RESULT[p][last]; m = gp == p; pr = np.array(LABELS)[(P * w).argmax(1)]
    rk = rank_metrics(P[:, 2], y[m])
    rows.append({"đề": p, "n": int(m.sum()), "n_Ineff": int((y[m] == "Ineffective").sum()),
                 "macroF1": f1_score(y[m], pr, average="macro", labels=LABELS, zero_division=0), **rk})
PP = pd.DataFrame(rows).set_index("đề").round(3); display(PP)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
PP[["PR_AUC", "base_rate"]].plot.bar(ax=ax[0]); ax[0].set_title("PR-AUC (Ineffective) so với mức đoán bừa"); ax[0].set_xticklabels([s[:14] for s in PP.index], rotation=35, ha="right", fontsize=7)
PP[["prec_top5", "base_rate"]].plot.bar(ax=ax[1]); ax[1].set_title("Precision trong top 5% đáng ngờ nhất"); ax[1].set_xticklabels([s[:14] for s in PP.index], rotation=35, ha="right", fontsize=7)
plt.tight_layout(); plt.savefig(FIGS / "step2_embed_ranking.png", dpi=150); plt.show()

## Step 2 · §9. Cách diễn giải (điền bằng lời của nhóm sau khi chạy)

* **Nếu S5/S6 hơn S3 rõ** (macro-F1 và PR-AUC cùng tăng, ở nhiều đề): embedding và ngữ cảnh nắm được tín hiệu mà đặc trưng tay bỏ sót.
* **Nếu gần bằng nhau**: tín hiệu văn bản bị chặn bởi nhãn nhiễu (κ = 0,32) và tỉ lệ Ineffective rất thấp; Step 2 vẫn chưa dùng làm cờ cho giáo viên.
* **Nếu `prec_top5` chỉ vài % và `lift` gần 1–3**: gắn cờ top 5% vẫn phần lớn là báo nhầm, ghi nhận là *giới hạn*, không gọi là công cụ ưu tiên feedback.
* Nhắc: các dòng chỉnh trọng số có macro-F1 cao hơn chủ yếu do **đổi ngưỡng quyết định**, không phải do mô hình phân biệt tốt hơn; PR-AUC (không phụ thuộc ngưỡng) mới phản ánh khả năng phân biệt.